# Case Study: SyntheticWinterV2 Evaluation Workflow

This notebook documents the evaluation procedure for the SyntheticWinterV2 image translation system in a report-ready format.
The protocol covers preprocessing, A2B inference, quantitative assessment, and downstream consistency checks.

## Study Objective
Assess summer-to-winter image translation quality using controlled, reproducible evaluation settings across two model variants.

## Required Project Artifacts
Store all required files in a single Google Drive directory, for example: `/MyDrive/SyntheticWinterV2/`

Core scripts:
- `evaluate.py`
- `models.py`
- `dataset.py`
- `utils.py`
- `checkpoint_epoch_100.pt` or an alternative checkpoint such as `gens.pth.tar` / `genw.pth.tar`

Required evaluation inputs:
- `dataset/testA/`
- `dataset/testB/`

Optional repository folders (not mandatory to upload for this workflow):
- `dataset_512/`
- `eval_data_512/`
- `output/`

If only raw input datasets are provided, this notebook automatically performs resizing to the target resolution before evaluation.

## 1. Environment Initialization: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
print('Drive mounted')

## 2. Runtime Dependencies and Hardware Verification

Colab often includes PyTorch by default; this step ensures all evaluation-specific dependencies (SSIM/FID) are installed and confirms GPU availability.

In [ ]:
!pip -q install pytorch-fid scikit-image tqdm

import os
import sys
import torch

print('Torch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('No GPU detected. Switch Colab runtime to GPU.')

## 3. Project Root Resolution and Input Integrity Check

Update `PROJECT_ROOT` only if a different Google Drive directory name is used for this project.

In [ ]:
from pathlib import Path

CANDIDATES = [
    Path('/content/drive/MyDrive/SyntheticWinterV2'),
    Path('/content/drive/MyDrive/SyntheticWinter_V2'),
    Path('/content/drive/MyDrive/SyntheticWinter'),
]

PROJECT_ROOT = next((p for p in CANDIDATES if p.is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not find the project folder in Google Drive. Update CANDIDATES in this cell.')

print('Project root:', PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

required_paths = [
    PROJECT_ROOT / 'evaluate.py',
    PROJECT_ROOT / 'models.py',
    PROJECT_ROOT / 'dataset.py',
    PROJECT_ROOT / 'utils.py',
    PROJECT_ROOT / 'checkpoint_epoch_100.pt',
    PROJECT_ROOT / 'dataset' / 'testA_model_v1',
    PROJECT_ROOT / 'dataset' / 'testA_model_v2',
    PROJECT_ROOT / 'dataset' / 'testB',
]

missing = [str(p) for p in required_paths if not p.exists()]
if missing:
    print('Missing required paths:')
    for item in missing:
        print(' -', item)
else:
    print('All required files and folders are present.')

## 4. Evaluation Dataset Preparation (A2B, Filename-Matched Protocol)

This case study evaluates only **summer -> synthetic winter** (`A2B`) with two source configurations:
- **Legacy model (v1 tar):** `dataset/testA_model_v1`, resized to **256x256**
- **Proposed model (v2 checkpoint):** `dataset/testA_model_v2`, resized to **512x512**

For each configuration, `testB` is resized and deterministically remapped to `testA` filenames to enable paired PSNR/SSIM computation.
Pairing uses lexicographic ordering and retains only the first `min(len(testA), len(testB))` samples to maintain one-to-one correspondence.

In [ ]:
%cd $PROJECT_ROOT

from pathlib import Path
from evaluate import resize_folder, VALID_EXTS
import shutil

# Define evaluation configurations for legacy and proposed model variants
OLD_CHECKPOINT = 'genw.pth.tar'            # legacy A2B checkpoint (summer -> synthetic winter)
NEW_CHECKPOINT = 'checkpoint_epoch_100.pt' # proposed epoch-based checkpoint

DATASET_DIR = PROJECT_ROOT / 'dataset'
COMMON_TEST_B = DATASET_DIR / 'testB'

OLD_SETUP = {
    'name': 'old_256',
    'img_size': 256,
    'source_testA': DATASET_DIR / 'testA_model_v1',
    'source_testB': COMMON_TEST_B,
    'prepared_root': PROJECT_ROOT / 'eval_data_256',
    'generated_dir': PROJECT_ROOT / 'output_old_256' / 'eval_fakeB',
    'metrics_out': PROJECT_ROOT / 'output_old_256' / 'eval_metrics_A2B.json',
}

NEW_SETUP = {
    'name': 'new_512',
    'img_size': 512,
    'source_testA': DATASET_DIR / 'testA_model_v2',
    'source_testB': COMMON_TEST_B,
    'prepared_root': PROJECT_ROOT / 'eval_data_512',
    'generated_dir': PROJECT_ROOT / 'output_new_512' / 'eval_fakeB',
    'metrics_out': PROJECT_ROOT / 'output_new_512' / 'eval_metrics_A2B.json',
}

def _list_images(folder: Path):
    return sorted([p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in VALID_EXTS])

def _clear_files(folder: Path):
    folder.mkdir(parents=True, exist_ok=True)
    for p in folder.iterdir():
        if p.is_file():
            p.unlink()

# Validate inputs, prepare resized evaluation subsets, and enforce deterministic filename pairing
for setup in (OLD_SETUP, NEW_SETUP):
    if not setup['source_testA'].is_dir():
        raise FileNotFoundError(f"Missing source folder: {setup['source_testA']}")
    if not setup['source_testB'].is_dir():
        raise FileNotFoundError(f"Missing source folder: {setup['source_testB']}")

    prep_a = setup['prepared_root'] / 'testA'
    prep_b = setup['prepared_root'] / 'testB'
    prep_b_pool = setup['prepared_root'] / '_testB_pool'

    setup['generated_dir'].mkdir(parents=True, exist_ok=True)
    setup['metrics_out'].parent.mkdir(parents=True, exist_ok=True)

    _clear_files(prep_a)
    _clear_files(prep_b)
    _clear_files(prep_b_pool)

    n_a = resize_folder(setup['source_testA'], prep_a, setup['img_size'])
    n_b = resize_folder(setup['source_testB'], prep_b_pool, setup['img_size'])

    a_files = _list_images(prep_a)
    b_pool_files = _list_images(prep_b_pool)
    matched_count = min(len(a_files), len(b_pool_files))

    if matched_count == 0:
        raise RuntimeError(f"No images available to match for {setup['name']}")

    # Copy winter reference images into testB using summer filenames for paired PSNR/SSIM
    for a_img, b_img in zip(a_files[:matched_count], b_pool_files[:matched_count]):
        shutil.copy2(b_img, prep_b / a_img.name)

    print(f"Prepared {setup['name']} ({setup['img_size']}x{setup['img_size']})")
    print('  source testA:', setup['source_testA'])
    print('  source testB:', setup['source_testB'])
    print('  prepared testA:', prep_a, f'({n_a} images)')
    print('  resized winter pool:', prep_b_pool, f'({n_b} images)')
    print('  matched testB:', prep_b, f'({matched_count} images)')
    print('  generated_dir:', setup['generated_dir'])
    print('  metrics_out:', setup['metrics_out'])

for ckpt_name in (OLD_CHECKPOINT, NEW_CHECKPOINT):
    ckpt_path = PROJECT_ROOT / ckpt_name
    print(f'Checkpoint {ckpt_name}:', 'FOUND' if ckpt_path.exists() else 'MISSING')

## 5. Batch Inference Execution (A2B Direction)

This step executes both evaluation tracks sequentially (legacy 256 and proposed 512) using the prepared, filename-matched datasets.

In [ ]:
import shlex

def run_eval(setup, checkpoint_name):
    ckpt_path = PROJECT_ROOT / checkpoint_name
    if not ckpt_path.exists():
        raise FileNotFoundError(f'Checkpoint not found: {ckpt_path}')

    cmd = (
        'python evaluate.py '
        f"--project_root . "
        f"--dataset_root {shlex.quote(str(setup['prepared_root']))} "
        f"--checkpoint {shlex.quote(checkpoint_name)} "
        '--direction A2B '
        f"--img_size {setup['img_size']} "
        '--no_resize '
        f"--generated_dir {shlex.quote(str(setup['generated_dir']))} "
        f"--metrics_out {shlex.quote(str(setup['metrics_out']))}"
    )

    print(f"\n=== Running {setup['name']} ({checkpoint_name}) ===")
    print(cmd)
    get_ipython().system(cmd)

run_eval(OLD_SETUP, OLD_CHECKPOINT)
run_eval(NEW_SETUP, NEW_CHECKPOINT)

## 6. Quantitative Output Audit and Metric Retrieval

This summary verifies generated output counts and loads key metrics for both runs:
- legacy tar checkpoint at 256x256
- proposed epoch checkpoint at 512x512

In [ ]:
import json

def show_result(setup, ckpt_name):
    gen_dir = setup['generated_dir']
    metrics_file = setup['metrics_out']
    count = len(list(gen_dir.glob('*'))) if gen_dir.exists() else 0

    print(f"\n=== {setup['name']} ({ckpt_name}) ===")
    print('Generated dir:', gen_dir)
    print('Generated images:', count)
    print('Metrics file exists:', metrics_file.exists())

    if metrics_file.exists():
        data = json.loads(metrics_file.read_text(encoding='utf-8'))
        print('Direction:', data.get('direction'))
        print('Image size:', data.get('img_size'))
        print('FID:', data.get('fid'))
        psnr_ssim = data.get('psnr_ssim')
        if psnr_ssim is None:
            print('PSNR/SSIM: skipped (no filename-matched pairs)')
        else:
            print('PSNR mean:', psnr_ssim.get('psnr_mean'))
            print('SSIM mean:', psnr_ssim.get('ssim_mean'))

show_result(OLD_SETUP, OLD_CHECKPOINT)
show_result(NEW_SETUP, NEW_CHECKPOINT)

In [ ]:
import json
import pandas as pd
import shlex
import subprocess

# Load FID scores from both evaluation outputs
fid_old = float(json.loads((OLD_SETUP['metrics_out']).read_text())['fid'])
fid_new = float(json.loads((NEW_SETUP['metrics_out']).read_text())['fid'])

# Retrieve downstream metrics from memory if available, otherwise from disk
downstream_data = globals().get('downstream_data', None)
downstream_summary_path = PROJECT_ROOT / 'downstream_eval' / 'summary.json'

if downstream_data is None and downstream_summary_path.exists():
    downstream_data = json.loads(downstream_summary_path.read_text(encoding='utf-8'))

# If unavailable, execute downstream validation to populate summary metrics
if downstream_data is None:
    print('Downstream summary not found. Running downstream validation now...')
    cmd_downstream = (
        'python downstream_validation.py '
        f"--summer_dir {shlex.quote(str(NEW_SETUP['source_testA']))} "
        f"--fake_old_dir {shlex.quote(str(OLD_SETUP['generated_dir']))} "
        f"--fake_new_dir {shlex.quote(str(NEW_SETUP['generated_dir']))} "
        f"--out_dir {shlex.quote(str(PROJECT_ROOT / 'downstream_eval'))} "
        '--score_thresh 0.5'
    )
    print(cmd_downstream)
    result = subprocess.run(cmd_downstream, shell=True, cwd=PROJECT_ROOT, capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print('STDERR:', result.stderr)

    if downstream_summary_path.exists():
        downstream_data = json.loads(downstream_summary_path.read_text(encoding='utf-8'))

downstream_old = downstream_data.get('old_256', {}) if downstream_data else {}
downstream_new = downstream_data.get('new_512', {}) if downstream_data else {}

summary_df = pd.DataFrame([
    {
        'Metric': 'FID Score',
        'old_256': f"{fid_old:.2f}",
        'new_512': f"{fid_new:.2f}",
        'Better': 'new_512' if fid_new < fid_old else 'old_256',
    },
    {
        'Metric': 'Downstream: Mean abs count diff',
        'old_256': f"{downstream_old['mean_abs_count_diff']:.4f}" if downstream_old else 'N/A',
        'new_512': f"{downstream_new['mean_abs_count_diff']:.4f}" if downstream_new else 'N/A',
        'Better': (
            'new_512'
            if downstream_old and downstream_new and downstream_new['mean_abs_count_diff'] < downstream_old['mean_abs_count_diff']
            else ('old_256' if downstream_old and downstream_new else 'N/A')
        ),
    },
    {
        'Metric': 'Downstream: Mean class L1 diff',
        'old_256': f"{downstream_old['mean_class_l1_diff']:.4f}" if downstream_old else 'N/A',
        'new_512': f"{downstream_new['mean_class_l1_diff']:.4f}" if downstream_new else 'N/A',
        'Better': (
            'new_512'
            if downstream_old and downstream_new and downstream_new['mean_class_l1_diff'] < downstream_old['mean_class_l1_diff']
            else ('old_256' if downstream_old and downstream_new else 'N/A')
        ),
    },
    {
        'Metric': 'Downstream: Mean confidence delta',
        'old_256': f"{downstream_old['mean_conf_delta']:.4f}" if downstream_old else 'N/A',
        'new_512': f"{downstream_new['mean_conf_delta']:.4f}" if downstream_new else 'N/A',
        'Better': 'closer to 0',
    },
    {
        'Metric': 'Downstream: Mean retention ratio',
        'old_256': f"{downstream_old['mean_retention_ratio']:.4f}" if downstream_old else 'N/A',
        'new_512': f"{downstream_new['mean_retention_ratio']:.4f}" if downstream_new else 'N/A',
        'Better': 'closer to 1.0',
    },
])

print('\n=== FINAL COMPARISON SUMMARY ===\n')
print(summary_df.to_string(index=False))
if downstream_data is None:
    print('\nDownstream metrics: N/A (downstream validation failed or did not produce summary.json).')
print('\nNote: Lower FID is better. Lower count-diff/L1-diff is better (better scene preservation).')

## 9. Consolidated Case-Study Summary Table

This section presents a consolidated comparison of both models across FID and downstream consistency indicators.

In [ ]:
out_dir_comparison = PROJECT_ROOT / 'comparison_outputs'

cmd_comparison = (
    'python create_comparison_panels.py '
    f"--summer_dir {shlex.quote(str(NEW_SETUP['source_testA']))} "
    f"--fake_old_dir {shlex.quote(str(OLD_SETUP['generated_dir']))} "
    f"--fake_new_dir {shlex.quote(str(NEW_SETUP['generated_dir']))} "
    f"--out_dir {shlex.quote(str(out_dir_comparison))}"
)

print('Creating comparison panels...')
print(cmd_comparison)
result = subprocess.run(cmd_comparison, shell=True, cwd=PROJECT_ROOT, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)

triplets_dir = out_dir_comparison / 'triplets'
if triplets_dir.exists():
    triplet_count = len(list(triplets_dir.glob('*')))
    print(f'\nCreated {triplet_count} comparison triplets in {out_dir_comparison}')

## 8. Qualitative Comparison Panel Generation

Generate side-by-side and triplet visual panels to support qualitative comparison on filename-matched samples.

In [ ]:
import subprocess
import json

# Ensure detector-related dependencies are available in the runtime
!pip -q install torchvision

out_dir_downstream = PROJECT_ROOT / 'downstream_eval'

cmd_downstream = (
    'python downstream_validation.py '
    f"--summer_dir {shlex.quote(str(NEW_SETUP['source_testA']))} "
    f"--fake_old_dir {shlex.quote(str(OLD_SETUP['generated_dir']))} "
    f"--fake_new_dir {shlex.quote(str(NEW_SETUP['generated_dir']))} "
    f"--out_dir {shlex.quote(str(out_dir_downstream))} "
    '--score_thresh 0.5'
)

print('Running downstream validation...')
print(cmd_downstream)
result = subprocess.run(cmd_downstream, shell=True, cwd=PROJECT_ROOT, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)

if (out_dir_downstream / 'summary.json').exists():
    downstream_data = json.loads((out_dir_downstream / 'summary.json').read_text(encoding='utf-8'))
    print('\nDownstream validation complete!')
    print('Summary saved to:', out_dir_downstream / 'summary.json')
else:
    print('Downstream validation skipped or failed.')
    downstream_data = None